# Push clusters into the viewer, refine them there, pull them back

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/python_package/f_notebooks_tutorials/43_push_clusters_refine_and_pull.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Compute clusters and a score with Scanpy, show them in the viewer with
`viewer.push(...)`, fix the clusters by eye (rename, merge), and bring
the refined labels back as an `obs` column. Cellucid becomes one step
of the pipeline, between clustering and annotation.

**Time:** 15 minutes. **You need:** `pip install cellucid scanpy
igraph leidenalg`. The same tutorial in R (Seurat `FindClusters`):
[Push clusters, refine, pull back in R](../../r_package/f_notebooks_tutorials/09_push_clusters_refine_and_pull.ipynb).

In [ ]:
import importlib.util

if importlib.util.find_spec("cellucid") is None:
    %pip install --quiet cellucid
if importlib.util.find_spec("scanpy") is None or importlib.util.find_spec("leidenalg") is None:
    %pip install --quiet scanpy igraph leidenalg

## The data

The 2,531-cell mouse pancreas (endocrinogenesis, day 15.5) AnnData checked
into the cellucid-python repository: UMAP in `obsm['X_umap_2d']`, cell types
in `obs['clusters']`, 27,998 genes. In a clone of the repository the cell
below finds it; anywhere else (Colab) it downloads it once. Use your own
`adata` in its place.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import anndata as ad

import cellucid

NAME = "endocrinogenesis_day15.5_raw.h5ad"
URL = (
    "https://raw.githubusercontent.com/theislab/cellucid-python/main/"
    "docs/user_guide/python_package/f_notebooks_tutorials/datasets/" + NAME
)


def pancreas_h5ad() -> Path:
    """The Pancreas file of a cellucid-python clone, else a copy downloaded once (35 MB)."""
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for candidate in (base / "datasets" / NAME, base / "docs/user_guide/python_package/f_notebooks_tutorials/datasets" / NAME):
            if candidate.is_file():
                return candidate
    if not (here / NAME).is_file():
        urlretrieve(URL, here / NAME)
    return here / NAME


DATA_PATH = pancreas_h5ad()
adata = ad.read_h5ad(DATA_PATH)
adata

## Cluster and score with Scanpy

On a copy, so `adata` keeps its counts. The copy has the same cells
in the same order, which is all `push` and `pull` need.

In [ ]:
import scanpy as sc

work = adata.copy()
sc.pp.normalize_total(work)
sc.pp.log1p(work)
sc.pp.highly_variable_genes(work, n_top_genes=2000)
sc.pp.pca(work, n_comps=30)
sc.pp.neighbors(work)
sc.tl.leiden(work, resolution=1.0, flavor="igraph", n_iterations=2, directed=False)
sc.tl.score_genes(work, ["Ins1", "Ins2", "Iapp", "Pdx1"], score_name="beta_score")
work.obs["leiden"].value_counts().sort_index()

## Show and push

`push` sends a column at once: a categorical (`category`, strings,
`bool`) becomes a categorical field, numbers a continuous field. A
`pandas.Series` is aligned by its index (obs names). Pushing the same
key again replaces the field.

Run the push cell once the viewer shows the cells: a column is checked
against the dataset the viewer opened, so before that `push` raises
and says to run it again.

In [ ]:
DATASET_ID = "pancreas-e15-5"

viewer = cellucid.show_anndata(DATA_PATH, dataset_name="Pancreas E15.5", dataset_id=DATASET_ID)

In [ ]:
viewer.push(work.obs["leiden"])                 # field "leiden"
viewer.push(work.obs["beta_score"])             # field "beta_score"
viewer.set_color_by("leiden")

## Refine in the viewer

Colour by `leiden` and compare with `clusters`. Then:

1. **Copy** the `leiden` field and **Rename** the copy `cell_type`.
2. In the legend of `cell_type`, rename each cluster to a cell type
   (the pencil, or double-click a label), and drag clusters of the same
   type onto each other to merge them.
3. Optional: lasso cells that are mislabelled and add them to a
   highlight group; colour by `beta_score` to check the Beta cells.

Then run the next cell.

In [ ]:
summary = viewer.pull(adata)
summary.columns, summary.skipped

`cell_type` is your field, so it comes back. `leiden` and
`beta_score` were pushed from this notebook, so they are skipped
(`summary.skipped` says so): `work` has them already. Pass
`include_pushed=True` to bring them back too — for example after you
edited `leiden` itself rather than a copy.

In [ ]:
import pandas as pd

if "cell_type" in adata.obs:
    print(pd.crosstab(work.obs["leiden"], adata.obs["cell_type"]))

Pushing again replaces the field in the viewer: recluster at another
resolution and push it under the same key, refine, and pull again.

In [ ]:
sc.tl.leiden(work, resolution=0.5, flavor="igraph", n_iterations=2, directed=False, key_added="leiden")
viewer.push(work.obs["leiden"])

In [ ]:
viewer.stop()

## Next

- [Gene sets round trip](45_gene_sets_round_trip.ipynb): gene sets
  into the viewer, signature scores back.
- [Export a subset and re-analyse it](46_export_an_h5ad_subset_and_reanalyse.ipynb)
- R: [Push clusters, refine, pull back in R](../../r_package/f_notebooks_tutorials/09_push_clusters_refine_and_pull.ipynb)